<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Feature Detection and Object Tracking — Algorithm</b></h1></div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Input video, initial object bounding box, ORB and RANSAC parameters |
| **Final outputs** | Per-frame tracking records, transformed object boxes, match/inlier metrics, five diagnostic figures |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment and open the tracking lab in VS Code.
2. Use this notebook as the exact coding order for `main.ipynb`.
3. Keep the first frame fixed as the reference throughout the sequence.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Paths and Discover the Video

**Purpose**

Select a reproducible video source.

**Inputs**

Supported videos under the lab data directory.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Create repository-local data and output paths.
3. Discover supported videos recursively and sort them deterministically.
4. Select the intended video by index and fail if no valid video exists.

**Outputs**

Sorted candidates; VIDEO_PATH; OUTPUT_DIR.

**Validation**

At least one candidate; VIDEO_INDEX identifies the intended sequence.

**On failure**

Stop if no supported file exists or selection is out of range.

## 2. Define the Initial Object Region and Parameters

**Purpose**

Specify the object geometry in reference-frame coordinates.

**Inputs**

ROW=24, COL=46, HEIGHT=170, WIDTH=160.

**Procedure**

1. Define row, column, height, and width of the reference object.
2. Convert them into a four-corner float32 bounding box shaped `(4,1,2)`.
3. Define maximum ORB features, RANSAC threshold, minimum matches, and minimum inliers.
4. Validate the bounding-box shape before opening the sequence.

**Outputs**

float32 initial_bounding_box shaped (4,1,2); ORB/RANSAC settings.

**Validation**

1000 features; threshold 5 pixels; minimum 4 matches and 4 inliers. Review ROI bounds against the decoded frame.

**On failure**

Correct an invalid ROI before extracting object features.

## 3. Initialize Feature Detection and Matching

**Purpose**

Use a distance suited to binary ORB descriptors.

**Inputs**

MAX_FEATURES and matching configuration.

**Procedure**

1. Create ORB with the configured feature limit.
2. Create a brute-force matcher with Hamming distance for binary ORB descriptors.
3. Enable `crossCheck=True` to retain reciprocal matches.
4. Keep the detector and matcher fixed for all frames.

**Outputs**

ORB detector; BFMatcher with Hamming and crossCheck.

**Validation**

Reciprocal matches compare binary descriptors; detector/matcher remain fixed.

**On failure**

Do not substitute Euclidean descriptor distance without changing the descriptor method.

## 4. Open and Validate the Reference Frame

**Purpose**

Establish a fixed reference view.

**Inputs**

VIDEO_PATH.

**Procedure**

1. Open the selected video with `VideoCapture` and verify it opened.
2. Read reported frame count when available.
3. Read the first frame and stop if decoding fails.
4. Convert the reference frame to grayscale.

**Outputs**

First decoded BGR frame, grayscale frame, reported frame count.

**Validation**

Capture opens and first frame decodes; metadata may be unavailable.

**On failure**

Release capture and stop on reference-frame failure.

## 5. Extract Reference Features Inside the Object

**Purpose**

Restrict reference features to the object.

**Inputs**

Reference grayscale frame and ROI mask.

**Procedure**

1. Create a zero mask matching the reference-frame size.
2. Fill only the initial object rectangle with 255.
3. Run ORB `detectAndCompute` using that mask.
4. Require at least four descriptors and descriptor width 32.
5. Preserve these reference keypoints/descriptors for the full sequence.

**Outputs**

Reference keypoints and descriptors shaped (N,32).

**Validation**

At least 4 descriptors; nonempty ROI within the image.

**On failure**

Stop on insufficient reference features; changing the ROI requires rerunning extraction.

## 6. Save the Reference Diagnostic

**Purpose**

Verify the reference region visually.

**Inputs**

Reference frame, box and keypoints.

**Procedure**

1. Draw the initial object boundary on a reference-frame copy.
2. Draw the reference ORB keypoints.
3. Convert BGR to RGB for Matplotlib.
4. Save `reference_orb_keypoints.png`.

**Outputs**

reference_orb_keypoints.png.

**Validation**

Draw on a copy; convert BGR to RGB; keypoints correspond to stored descriptors.

**On failure**

Correct ROI or channel display errors before tracking.

## 7. Implement Frame Matching and Homography Estimation

**Purpose**

Estimate the reference-to-current mapping robustly.

**Inputs**

Current grayscale frame and fixed reference features.

**Procedure**

1. Detect ORB keypoints/descriptors in the current grayscale frame.
2. If descriptors are missing or too few, return a structured failure record.
3. Match reference descriptors to current descriptors and sort by distance.
4. If fewer than four matches remain, return failure.
5. Build paired reference/current point arrays.
6. Estimate the homography with RANSAC.
7. Reject missing, non-finite, or wrongly shaped homographies.
8. Count inliers and reject results below the minimum.
9. Return success state, homography, inlier mask, keypoints, matches, and counts.

**Outputs**

Structured success/failure record, H, matches and inlier mask.

**Validation**

At least 4 matches/inliers; finite 3 by 3 H; inlier count cannot exceed match count.

**On failure**

Return the existing failure reason for missing features, matches, H or inliers; do not invent a transform.

## 8. Track Through the Complete Video

**Purpose**

Record both successful and failed frame estimates.

**Inputs**

Video frames after the reference; matching helper.

**Procedure**

1. Start from the frame immediately after the reference frame.
2. For each frame: convert to grayscale and call the matching helper.
3. On success, transform the four reference-box corners through the homography.
4. Store frame index, homography, transformed box, match count, and inlier count.
5. On failure, store frame index and failure reason instead of inventing a box.
6. Continue until the video ends, then release the capture.

**Outputs**

Frame-indexed tracking records with transformed boxes on success.

**Validation**

Each successful box is finite (4,1,2); failed frames remain represented.

**On failure**

Continue after a recorded tracking failure; release capture after decoding ends.

## 9. Compute Sequence Metrics

**Purpose**

Quantify success over the processed sequence.

**Inputs**

All tracking records, including failures.

**Procedure**

1. Extract successful tracking records.
2. Stop if no valid homography was produced.
3. Create arrays of frame indices, matches, inliers, and inlier ratios.
4. Compute tracking success rate and summary statistics.

**Outputs**

Success rate; per-frame match/inlier/ratio arrays.

**Validation**

Success denominator is all processed frames; ratios stay in [0,1]; at least one valid result.

**On failure**

Stop if every frame fails; do not compute success rate on successful frames alone.

## 10. Implement Random-Access Frame Reading

**Purpose**

Retrieve frames for diagnostics without altering tracking records.

**Inputs**

Video path and recorded frame index.

**Procedure**

1. Create a helper that reopens the video and seeks to a requested frame index.
2. Read exactly one frame, release the capture, and fail if decoding fails.
3. Use this helper only for visualization; do not rerun tracking state.

**Outputs**

One decoded frame.

**Validation**

Requested index corresponds to the stored result; capture is released.

**On failure**

Raise on failed random-access decoding rather than using another frame.

## 11. Visualize Representative Tracking Frames

**Purpose**

Show tracking quality across time.

**Inputs**

Up to six successful records spread over the sequence.

**Procedure**

1. Choose up to six successful results uniformly across the sequence.
2. Read each corresponding frame.
3. Draw the stored transformed bounding box and annotate frame/inlier information.
4. Save the representative tracking grid.

**Outputs**

representative_tracking_frames.png.

**Validation**

Draw stored boxes on the corresponding frames; label frame and inlier counts.

**On failure**

Report unavailable frames; successful-frame sampling alone does not represent failures.

## 12. Visualize RANSAC Inlier Matches

**Purpose**

Show which descriptor matches support a homography.

**Inputs**

A successful middle frame and fixed reference.

**Procedure**

1. Choose a successful frame near the middle of the sequence.
2. Recompute descriptor matching for that frame to obtain match objects.
3. Use the RANSAC mask so only geometrically consistent matches are highlighted.
4. Draw reference/current boxes and save the inlier-match diagnostic.

**Outputs**

ransac_inlier_matches.png.

**Validation**

Current implementation reruns matching for this diagnostic and draws its inliers.

**On failure**

If the rerun fails, report it; stochastic RANSAC reruns need not reproduce the stored inlier set exactly.

## 13. Plot Match and Inlier Evolution

**Purpose**

Expose weak frames through temporal diagnostics.

**Inputs**

Recorded frame indices, matches, inliers and ratios.

**Procedure**

1. Plot descriptor-match and RANSAC-inlier counts against frame index.
2. Plot inlier ratio against frame index separately.
3. Use the curves to expose weak frames even when a box exists.
4. Save both figures.

**Outputs**

Two temporal plots.

**Validation**

Preserve sequence indices and failure records; verify 0<=inliers<=matches.

**On failure**

Investigate gaps or impossible counts rather than smoothing them away.

## 14. Run Final Validation

**Purpose**

Check geometric records and required outputs.

**Inputs**

Reference descriptors, sequence records, video metadata and five files.

**Procedure**

1. Verify reference descriptors exist, have enough rows, and width 32.
2. Verify processed-frame count is consistent with video metadata when available.
3. For every success verify a finite 3×3 homography and finite `(4,1,2)` box.
4. Verify match/inlier counts and ratios are valid.
5. Verify all declared output files exist before printing the pass message.

**Outputs**

Final validation outcome.

**Validation**

N by 32 descriptors; finite 3 by 3 H and (4,1,2) boxes on success; valid counts and ratios.

**On failure**

Stop on failed invariants or missing figures; inspect metadata disagreements as decoding/container limitations.

## Complete End-to-End Pseudocode

This is control-flow pseudocode, not an additional Python API. It follows the current implementation; review requirements and known gaps are distinguished below.

```text
discover sorted videos; choose VIDEO_INDEX
set reference rectangle, ORB limit, Hamming cross-check and RANSAC settings
open video; decode first frame; stop if reference cannot be read
mask reference ROI; detect ORB keypoints/descriptors
require at least 4 reference descriptors of width 32
save reference diagnostic
for each later decoded frame:
    detect current features
    if insufficient descriptors: record failure; continue
    match fixed reference descriptors; sort matches by Hamming distance
    if fewer than 4 matches: record failure; continue
    estimate reference-to-current homography with RANSAC
    if missing/nonfinite H or fewer than 4 inliers: record failure; continue
    transform original four box corners through H
    store frame index, H, box, match and inlier counts
release capture; require at least one successful estimate
compute success rate over all processed records and per-frame ratios
retrieve representative successful frames; draw stored boxes
rerun matching on a middle successful frame for the inlier-match diagnostic
plot recorded matches, inliers and ratios versus original frame index
validate descriptors, frame counts, successful geometry and five figures
```

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Validate the Input Video and Output Paths
- **Task 2:** Define the Initial Bounding Box and Tracking Parameters
- **Task 3:** Initialize ORB and the Hamming-Distance Matcher
- **Task 4:** Read and Validate the Reference Frame
- **Task 5:** Detect Reference ORB Features Inside the Object Region
- **Task 6:** Visualize the Reference Object and ORB Keypoints
- **Task 7:** Define Frame Matching and RANSAC Homography Estimation
- **Task 8:** Track the Object Throughout the Video
- **Task 9:** Compute Tracking Summary Metrics
- **Task 10:** Visualize Representative Tracking Frames
- **Task 11:** Visualize RANSAC Inlier Matches
- **Task 12:** Analyze Matches, Inliers, and Inlier Ratio Across the Sequence
- **Task 13:** Run Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation**.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 5 files under `../outputs/figures/`:

- `reference_orb_keypoints.png`
- `representative_tracking_frames.png`
- `ransac_inlier_matches.png`
- `matches_and_inliers_by_frame.png`
- `inlier_ratio_by_frame.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/computer-vision/feature-tracking/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `match_features_and_estimate_homography`
- `read_video_frame_at_index`

## Implementation Limits and Interpretation

- Reference descriptors remain fixed to the first frame; this is reference-to-frame homography tracking, not optical flow or a frame-to-frame descriptor update.
- The inlier-match figure reruns matching/RANSAC. It is a diagnostic rerun and may differ from the stored sequence estimate; the sequence metrics still use recorded tracking results.
- Four matches/inliers are necessary minima for this implementation, not sufficient proof of a well-conditioned mapping. Finite boxes and high inlier ratio do not establish localization accuracy without ground truth.
- Report decoding/frame-count limitations separately from recorded tracking failures.

## 15. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.